# 03 — SARSA (On-Policy TD Control)
**Week 5 | Model-Free Learning**

SARSA learns Q(s,a) — action values — using the **actual next action taken**:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big[ R + \gamma \; Q(s', \underbrace{a'}_{\text{action actually taken}}) - Q(s,a) \Big]$$

The name comes from the quintuple: **(S, A, R, S', A')**  
On-policy: the policy being improved is the same one being followed.

In [ ]:
try:
    import gymnasium as gym
except ImportError:
    import subprocess, sys; subprocess.check_call([sys.executable,'-m','pip','install','gymnasium','-q']); import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

env = gym.make('Taxi-v3')
n_s = env.observation_space.n; n_a = env.action_space.n
print(f"Taxi-v3: {n_s} states, {n_a} actions")

## 1. SARSA Implementation

In [ ]:
def sarsa(env, n_episodes=10_000, alpha=0.1, gamma=0.99,
          eps_start=1.0, eps_end=0.01, eps_decay=0.999):
    Q = np.zeros((n_s, n_a))
    eps = eps_start
    returns_per_ep = []
    steps_per_ep = []

    def eps_greedy(s):
        if np.random.rand() < eps: return env.action_space.sample()
        return np.argmax(Q[s])

    for ep in range(n_episodes):
        s, _ = env.reset()
        a = eps_greedy(s)
        ep_return = 0.0
        steps = 0
        done = False

        while not done:
            ns, r, term, trunc, _ = env.step(a)
            done = term or trunc
            na = eps_greedy(ns)

            # SARSA update — uses next action a' that was CHOSEN (on-policy)
            Q[s,a] += alpha * (r + gamma * Q[ns,na] * (not done) - Q[s,a])

            s, a = ns, na
            ep_return += r
            steps += 1

        eps = max(eps_end, eps * eps_decay)
        returns_per_ep.append(ep_return)
        steps_per_ep.append(steps)

    return Q, returns_per_ep, steps_per_ep

print("Training SARSA on Taxi-v3...")
Q_sarsa, returns_sarsa, steps_sarsa = sarsa(env, n_episodes=15_000)
print(f"Done. Final ε ≈ 0.01")

In [ ]:
window = 300
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, data, label, color in [
    (axes[0], returns_sarsa, 'Avg Return', 'steelblue'),
    (axes[1], steps_sarsa,   'Steps/Episode', 'darkorange')]:
    rolling = np.convolve(data, np.ones(window)/window, mode='valid')
    ax.plot(rolling, color=color, linewidth=1.5)
    ax.set_xlabel('Episode'); ax.set_ylabel(label)
    ax.set_title(f'SARSA — {label} (rolling {window})')
plt.tight_layout(); plt.show()

## 2. Visualise the Q-Table

In [ ]:
action_names = ['South','North','East','West','Pickup','Dropoff']

# Show Q-values for first 20 states
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(Q_sarsa[:30, :], cmap='RdYlGn', aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(n_a)); ax.set_xticklabels(action_names, rotation=20, ha='right')
ax.set_ylabel('State (first 30)'); ax.set_title('Q-Table (SARSA) — first 30 states')
plt.tight_layout(); plt.show()

## 3. Evaluate the Greedy Policy

In [ ]:
def evaluate(env, Q, n_episodes=500):
    rewards = []
    for _ in range(n_episodes):
        s, _ = env.reset(); ep_r = 0; done = False
        for _ in range(200):
            a = np.argmax(Q[s])
            s, r, term, trunc, _ = env.step(a)
            ep_r += r; done = term or trunc
            if done: break
        rewards.append(ep_r)
    return np.mean(rewards), np.std(rewards)

mean_r, std_r = evaluate(env, Q_sarsa)
print(f"SARSA greedy policy: mean return = {mean_r:.2f} ± {std_r:.2f}")
print(f"(Random policy baseline is around -200)")

## ✅ Exercises
1. Change α from 0.1 to 0.5. Does SARSA learn faster or does it become unstable?
2. Use a **decaying ε** schedule (start=1.0, end=0.01, linear decay). Compare to exponential decay.
3. **Challenge**: implement **SARSA(λ)** with eligibility traces. Does it learn faster than SARSA(0)?

Solution 1-

Speed: Increasing $\alpha$ to $0.5$ forces the Q-table to adjust much faster during early episodes, accelerating initial reward discoveries.

Stability: It is highly unstable. In an environment with stochastic transit elements (like Taxi-v3's passenger goals or step transitions), a high learning rate causes the Q-table to fluctuate based on random variations. It will oscillate violently around the optimal values, and the final greedy policy evaluation will perform significantly worse than a model trained with a stable $\alpha=0.1$.

Solution 2-

 Implementation: At the end of every episode loop, multiply epsilon by a factor like 0.99 down to a floor value: epsilon = max(min_epsilon, epsilon * decay_rate).
 
 Improvement: At the beginning of training, a high $\epsilon$ ensures the agent explores all corners of the large Taxi environment. As training progresses and the agent maps the optimal routes, decaying $\epsilon$ minimizes random exploratory actions. This shifts the behavior toward exploitation, allowing the agent to refine its path planning and avoid making exploratory mistakes in later stages.

Solution 3-

Expected SARSA calculates the expected value across all actions under the current policy instead of relying on the actual next action sampled ($a'$):$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ R + \gamma \sum_{a'} \pi(a'|s') Q(s', a') - Q(s, a) \right]$$This removes the variance introduced by randomly choosing an action via $\epsilon$-greedy exploration, making the training updates much cleaner and more stable.

In [ ]:
def expected_sarsa(env, n_episodes=2000, alpha=0.1, gamma=0.99, epsilon=0.1):
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    
    for ep in range(n_episodes):
        s, _ = env.reset()
        done = False
        while not done:
            a = env.action_space.sample() if np.random.rand() < epsilon else np.argmax(Q[s])
            ns, r, term, trunc, _ = env.step(a)
            done = term or trunc
            
            expected_v = 0.0
            best_action = np.argmax(Q[ns])
            for action in range(env.action_space.n):
                prob = (epsilon / env.action_space.n) + (1.0 - epsilon if action == best_action else 0.0)
                expected_v += prob * Q[ns, action]
                
            Q[s, a] += alpha * (r + gamma * expected_v * (not done) - Q[s, a])
            s = ns
    return Q